# 06. Conditional quantile regression: crime and the price distribution

Inspired by the quantile analysis in [Kallberg and Shimizu (2026)](https://doi.org/10.1007/s11146-024-09997-w). This London adaptation uses MPS prior-twelve-completed-month counts. It is not an IV quantile model or a replication of the Seattle estimates.

Question: does the conditional crime coefficient differ across the 5th, 25th, 50th, 75th and 95th price quantiles? A conditional quantile is not a subsample obtained by sorting observed prices. Every quantile uses all analysis-sample sales.

Property, distance, economic, borough and year-quarter controls are included directly. Ordinary OLS/TWFE demeaning is **not** used for quantiles. Thousands of LSOA dummy coefficients would create a different, expensive panel-quantile problem; this exploratory notebook uses borough effects and cannot be interpreted as M7. Quantile regression does not remove crime endogeneity.


In [1]:
from pathlib import Path
try:
    from google.colab import drive
except ImportError:
    ROOT=Path.cwd()
else:
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/HE3027 Project')
DATA_DIR=ROOT/'data'
MERGED=DATA_DIR/'crime_housing_merged.csv.gz'
if not MERGED.is_file():raise FileNotFoundError('Run the data-audit notebook first.')

OUT=DATA_DIR/'06_crime_quantile_outputs';OUT.mkdir(exist_ok=True)
SEED=302705
MAX_SALES=6000  # Reproducible transaction subsample; None uses all eligible sales.
QUANTILES=[.05,.25,.50,.75,.95]
BOOTSTRAP_REPS=39  # Pilot inference; use 199+ for a final report.


In [2]:
import hashlib
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
CRIMES = ['allcrime', 'violent_crime', 'propertycrime']
pd.set_option('display.max_columns', 12)
pd.set_option('display.precision', 4)

def table(data, filename):
    """Save the complete table and display a compact preview."""
    data.to_csv(OUT / (filename + '.csv'))
    display(data.head(20) if len(data) > 20 else data)
    if len(data) > 20:
        print(f'Showing 20 of {len(data)} rows; the complete table is saved as {filename}.csv.')

VARIABLE_LABELS = {
    'detached_dum': 'Detached (relative to flat)',
    'semi_d_dum': 'Semi-detached (relative to flat)',
    'terrace_dum': 'Terraced (relative to flat)',
    'freehold': 'Freehold',
    'newbuild': 'New build',
    'tube_distnear_km': 'Distance to tube station (km)',
    'bus_distnear_km': 'Distance to bus stop (km)',
    'thamesriv_dist_km': 'Distance to Thames (km)',
    'Nearest_Park_dist_km': 'Distance to park (km)',
    'dist_to_cbd': 'Distance to CBD (km)',
    'Bus_stopscount_200m': 'Bus stops within 200m',
    'heritage_count_200m': 'Heritage buildings within 200m',
    'percent_noedu': 'Residents without qualifications',
    'percent_minorities': 'Minority residents',
    'lone_hh': 'Lone-parent households',
    'lnpay': 'Log borough annual pay',
    'jobdensity': 'Borough job density',
    'hoursworked': 'Borough weekly hours worked',
    'unemployment': 'Borough unemployment',
}

def coefficient_inference(coefficients, results):
    """Compute inference for every coefficient from its saved clustered SE."""
    keys = [name for name in ['panel', 'model', 'crime', 'measure', 'timing']
            if name in coefficients and name in results]
    metadata = results[keys + ['clusters']]
    enriched = coefficients.drop(columns=['clusters', 't', 'p', 'ci_low', 'ci_high'], errors='ignore')
    enriched = enriched.merge(metadata, on=keys, how='left', validate='many_to_one')
    if enriched.clusters.isna().any():
        raise ValueError('Coefficient rows could not be matched to regression metadata.')
    degrees_freedom = enriched.clusters - 1
    enriched['t'] = enriched.beta / enriched.se
    enriched['p'] = 2 * stats.t.sf(np.abs(enriched.t), degrees_freedom)
    critical = stats.t.ppf(0.975, degrees_freedom)
    enriched['ci_low'] = enriched.beta - critical * enriched.se
    enriched['ci_high'] = enriched.beta + critical * enriched.se
    return enriched

def show_models(results, filename, coefficient_filename):
    """Show one expandable table per model with crime AND control coefficients."""
    from html import escape
    from IPython.display import HTML

    results.to_csv(OUT / (filename + '.csv'), index=False)
    coefficients = coefficient_inference(pd.read_csv(OUT / coefficient_filename), results)
    coefficients.to_csv(OUT / coefficient_filename, index=False)
    cards = []

    def number(value):
        if pd.isna(value):
            return '—'
        if value != 0 and abs(value) < 0.0001:
            return f'{value:.2e}'
        return f'{value:.4f}'

    for _, regression in results.iterrows():
        selected = coefficients.model.eq(regression['model'])
        for key in ['panel', 'crime', 'measure', 'timing']:
            if key in coefficients and key in results:
                selected &= coefficients[key].eq(regression[key])
        rows = coefficients.loc[selected].copy()
        title = regression['model'] + ' · ' + str(regression.get('crime', regression.get('measure', '')))
        if 'timing' in results:
            title += ' · ' + regression['timing']
        rendered = []
        for _, row in rows.iterrows():
            label = VARIABLE_LABELS.get(row.variable, row.variable)
            if row.variable.startswith('log1p_'):
                label = 'Crime: log(1 + ' + row.variable[6:].replace('_', ' ') + ')'
            rendered.append({
                'Variable': label,
                'Coefficient': number(row.beta),
                'Clustered SE': number(row.se),
                't statistic': f'{row.t:.2f}',
                '95% CI': number(row.ci_low) + ' to ' + number(row.ci_high),
                'p-value': '<0.001' if row.p < 0.001 else f'{row.p:.3f}',
            })
        effects = regression.get('fixed_effects', '')
        effects = 'None' if pd.isna(effects) or not effects else str(effects)
        r2_label = 'Within R²' if regression.r2_type == 'within' else 'R²'
        subtitle = (f'N = {int(regression.N):,} | LSOA clusters = {int(regression.clusters):,}'
                    f' | {r2_label} = {regression.r2:.4f}')
        body = pd.DataFrame(rendered).to_html(index=False, border=0, escape=True)
        omitted = regression.get('absorbed_or_collinear', '')
        note = '' if pd.isna(omitted) or not omitted else '<p>Absorbed/collinear: ' + escape(str(omitted)) + '</p>'
        cards.append('<details class="reg-card"><summary>' + escape(title) + '</summary>'
                     '<p>' + escape(subtitle) + '</p><p>Fixed effects: ' + escape(effects) + '</p>'
                     + body + note + '</details>')
    style = '<style>.reg-card{margin:10px 0;padding:12px;border:1px solid #bbb;border-radius:6px;overflow-x:auto}.reg-card summary{cursor:pointer;font-weight:600}.reg-card table{border-collapse:collapse;width:100%;max-width:950px}.reg-card th,.reg-card td{padding:7px 10px;border-bottom:1px solid #ddd;text-align:right;white-space:nowrap}.reg-card th:first-child,.reg-card td:first-child{text-align:left}.reg-card p{font-size:13px}</style>'
    display(HTML(style + '<p>Expand a regression to view all estimated coefficients. Standard errors are clustered by LSOA.</p>' + ''.join(cards)))
    return coefficients


In [3]:
df=pd.read_csv(MERGED,low_memory=False)
assert df.transaction_id.is_unique


In [4]:
LOCATION_MODE = 'continuous'
RIVER_THRESHOLD_METRES = None

data = df.copy()
data['lnprice'] = np.log(data.price.where(data.price.gt(0)))
data['newbuild'] = data.age.eq('Y').astype(float)
data['grossannualpay_k'] = data.grossannualpay / 1000
data['Nearest_Park_dist_km'] = data['Nearest_Park_dist'] / 1000
PROPERTY = ['detached_dum','semi_d_dum','terrace_dum','freehold','newbuild']
ECONOMIC = ['grossannualpay_k','jobdensity','hoursworked','unemployment']
if LOCATION_MODE == 'continuous':
    LOCATION = []
    for name in ['tube_distnear','bus_distnear','thamesriv_dist']:
        data[name+'_km'] = data[name]/1000
        LOCATION.append(name+'_km')
    LOCATION += ['dist_to_cbd','Nearest_Park_dist_km']
elif LOCATION_MODE == 'indicators':
    if RIVER_THRESHOLD_METRES is None or RIVER_THRESHOLD_METRES <= 0:
        raise ValueError('Specify a positive river proximity threshold in metres.')
    thresholds = {'tube_distnear':200, 'bus_distnear':200, 'thamesriv_dist':RIVER_THRESHOLD_METRES}
    LOCATION = ['tube_near','bus_near','river_near']
    for name, target in zip(thresholds,LOCATION):
        data[target] = data[name].le(thresholds[name]).astype(float).where(data[name].notna())
else:
    raise ValueError('LOCATION_MODE must be continuous or indicators.')
VARIABLE_LABELS.update({'grossannualpay_k':'Borough annual pay (GBP 1,000)',
                        'tube_near':'Tube station within 200m', 'bus_near':'Bus stop within 200m',
                        'river_near':'River within specified threshold'})


In [5]:
MAIN=['mps_all','mps_violence','mps_property']
CRIME_NAMES=dict(zip(MAIN,['All crime','Violence','Property crime']))
CONTROLS=PROPERTY+LOCATION+[c for c in ECONOMIC if c!='jobdensity']
for c in MAIN:data['log1p_'+c]=np.log1p(data[c+'_prior12'])
required=['lnprice','lsoacode','msoa11','oslaua','yearquarter','year','lat','lon']+CONTROLS+['log1p_'+c for c in MAIN]
valid=data[required].replace('',np.nan).notna().all(axis=1)
valid &= np.isfinite(data[required].select_dtypes(include='number')).all(axis=1)
valid &= data.analysis_geography_ok & data.mps_exposures_available & data.mps_geography_ok
eligible=data.loc[valid].copy()

sample=eligible.sample(n=min(MAX_SALES,len(eligible)),random_state=SEED).sort_values('transaction_id') if MAX_SALES else eligible.sort_values('transaction_id')
sample=sample.copy()
sample[['transaction_id','lsoacode']].to_csv(OUT/'quantile_sample_ids.csv',index=False)
table(pd.DataFrame({'eligible':[len(eligible)],'analysed':[len(sample)],'lsoa_clusters':[sample.lsoacode.nunique()]}),'quantile_sample_size')
table(pd.concat([eligible.groupby('year').size().rename('eligible'),sample.groupby('year').size().rename('analysed')],axis=1),'sample_by_year')


,eligible,analysed,lsoa_clusters
0,338347,6000,3129


,eligible,analysed
year,,
2012,85375,1554
2013,99429,1724
2014,108207,1895
2015,45336,827


## 1. Estimate the conditional distribution directly

$$Q_\tau(\ln P_i\mid C_i,X_i,b,q)=\alpha_\tau+\beta_\tau\ln(1+C_i)+\gamma_\tau'X_i+\mu_{b,\tau}+\lambda_{q,\tau}.$$

The same reproducible transaction subsample is used for all categories and quantiles; the full eligible sample is larger. The cap is for exploratory runtime, not selected by results. LSOA pairs bootstrap resamples entire clusters of retained sales, holding the analysis subsample fixed. Its intervals account for within-LSOA dependence, not arbitrary dependence between neighbouring LSOAs. With 39 replications, percentile intervals are provisional and have substantial Monte Carlo uncertainty; no bootstrap p-values are reported.

IRLS estimates are checked for convergence. A sparse linear-programming solution is used when IRLS does not converge. Bootstrap fits use the exact HiGHS interior-point linear programme directly after one initial iteration; all fit methods, objective values and bootstrap success counts are saved. The solver is [SciPy HiGHS](https://docs.scipy.org/doc/scipy/reference/optimize.linprog-highs.html); the iterative implementation is [statsmodels QuantReg](https://www.statsmodels.org/stable/generated/statsmodels.regression.quantile_regression.QuantReg.html).


In [6]:
import warnings
from statsmodels.regression.quantile_regression import QuantReg
from scipy.optimize import linprog
from scipy import sparse
dummy=pd.get_dummies(sample[['oslaua','yearquarter']].astype(str),drop_first=True,dtype=float)
design=pd.concat([sample[CONTROLS].astype(float),dummy],axis=1)
design.insert(0,'Intercept',1.)
names=list(design.columns)
xbase=design.to_numpy(float)
base_scale=xbase.std(axis=0);base_scale[base_scale<1e-12]=1
xbase=xbase/base_scale
y=sample.lnprice.to_numpy(float)
group=pd.factorize(sample.lsoacode,sort=True)[0]
def fit_quantile(y,x,tau,max_iter=1500):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        fit=QuantReg(y,x).fit(q=tau,max_iter=max_iter,p_tol=1e-6)
    converged=fit.iterations<max_iter and not any('cycle' in str(w.message).lower() for w in caught)
    method='IRLS'
    beta=np.asarray(fit.params)
    if not converged:
        n,p=x.shape
        matrix=sparse.hstack([sparse.csr_matrix(x),sparse.eye(n),-sparse.eye(n)],format='csr')
        objective=np.r_[np.zeros(p),np.repeat(tau,n),np.repeat(1-tau,n)]
        result=linprog(objective,A_eq=matrix,b_eq=y,bounds=[(None,None)]*p+[(0,None)]*(2*n),method='highs-ipm',options={'time_limit':90})
        if not result.success:raise RuntimeError('Quantile LP failed: '+result.message)
        beta=result.x[:p];method='HiGHS IPM LP';converged=True
    residual=y-x@beta
    loss=np.sum(np.where(residual>=0,tau*residual,(tau-1)*residual))
    return beta,method,loss


In [7]:
point=[];all_coefficients=[];designs={}
for category in MAIN:
    crime=sample['log1p_'+category].to_numpy(float)
    scale=max(crime.std(),1e-12)
    x=np.column_stack([crime/scale,xbase]);scales=np.r_[scale,base_scale]
    designs[category]=(x,scales)
    for tau in QUANTILES:
        beta,method,loss=fit_quantile(y,x,tau)
        estimates=beta/scales
        point.append(dict(crime=category,quantile=tau,beta=estimates[0],N=len(sample),clusters=sample.lsoacode.nunique(),solver=method,pinball_loss=loss))
        all_coefficients.append(pd.DataFrame({'variable':['log1p_'+category]+names,'beta':estimates}).assign(crime=category,quantile=tau))
        print(category,'tau',tau,'beta',round(estimates[0],5),method,flush=True)
points=pd.DataFrame(point).assign(data_source='MPS', exposure='prior12', estimator='Quantile')
pd.concat(all_coefficients).to_csv(OUT/'all_quantile_coefficients.csv',index=False)
points.to_csv(OUT/'quantile_points.csv',index=False)


mps_all tau 0.05 beta -0.05663 IRLS
mps_all tau 0.25 beta -0.0624 IRLS
mps_all tau 0.5 beta -0.0484 IRLS
mps_all tau 0.75 beta -0.04312 IRLS
mps_all tau 0.95 beta 0.00968 IRLS
mps_violence tau 0.05 beta -0.1325 IRLS
mps_violence tau 0.25 beta -0.14746 IRLS
mps_violence tau 0.5 beta -0.13231 IRLS
mps_violence tau 0.75 beta -0.12671 IRLS
mps_violence tau 0.95 beta -0.08246 IRLS
mps_property tau 0.05 beta 0.00033 IRLS
mps_property tau 0.25 beta 0.00601 IRLS
mps_property tau 0.5 beta 0.01011 HiGHS IPM LP
mps_property tau 0.75 beta 0.02287 IRLS
mps_property tau 0.95 beta 0.07222 IRLS


## 2. Cluster bootstrap and matched-sample OLS benchmark


In [8]:
from linearmodels.iv import AbsorbingLS
def absorb(matrix, groups, tol=1e-10, max_iter=2000):
    z = np.asarray(matrix, dtype=float).copy()
    codes = [pd.factorize(g, sort=False)[0] for g in groups]
    if any(((g < 0).any() for g in codes)):
        raise ValueError('Missing fixed-effect ID')
    for iteration in range(max_iter):
        before = z.copy()
        for g in codes:
            counts = np.bincount(g)
            sums = np.column_stack([np.bincount(g, weights=z[:, j], minlength=len(counts)) for j in range(z.shape[1])])
            z -= (sums / counts[:, None])[g]
        if np.max(np.abs(z - before)) < tol:
            return (z, iteration + 1)
    raise RuntimeError('Fixed-effect absorption did not converge')
def fit_absorbed(data, crime, controls, fixed_effects, cluster='lsoacode'):
    """Estimate OLS with high-dimensional fixed effects using linearmodels."""
    names = ([crime] if crime is not None else []) + controls
    x = data[names].astype(float).copy()
    y = data.lnprice.astype(float).copy()
    scale = x.std(axis=0).to_numpy().copy()
    scale[scale < 1e-14] = 1
    x_scaled = x / scale
    if fixed_effects:
        absorbed = pd.DataFrame(
            {c: pd.Categorical(data[c]) for c in fixed_effects}, index=data.index
        )
    else:
        absorbed = None
        y = y - y.mean()
        x_scaled = x_scaled - x_scaled.mean(axis=0)
    fit = AbsorbingLS(
        y, x_scaled, absorb=absorbed, drop_absorbed=True
    ).fit(cov_type='clustered', clusters=data[cluster], debiased=True)
    group = pd.factorize(data[cluster], sort=False)[0]
    G = int(group.max() + 1)
    if G < 2:
        raise ValueError('Need multiple inference clusters')
    identified = list(fit.params.index)
    if crime is not None and crime not in identified:
        raise ValueError('Crime has no identifying variation in this specification')
    estimates = fit.params.to_numpy() / scale[[names.index(c) for c in identified]]
    se = fit.std_errors.to_numpy() / scale[[names.index(c) for c in identified]]
    critical = stats.t.ppf(0.975, G - 1)
    j = identified.index(crime) if crime is not None else 0
    result = {
        'N': len(data), 'clusters': G,
        'beta': estimates[j] if crime is not None else np.nan,
        'se': se[j] if crime is not None else np.nan,
        'p': 2 * stats.t.sf(abs(estimates[j] / se[j]), G - 1) if crime is not None else np.nan,
        'ci_low': estimates[j] - critical * se[j] if crime is not None else np.nan,
        'ci_high': estimates[j] + critical * se[j] if crime is not None else np.nan,
        'r2': fit.rsquared, 'r2_type': 'within' if fixed_effects else 'ordinary',
        'crime_residual_sd': x_scaled.iloc[:, names.index(crime)].std() * scale[names.index(crime)] if crime is not None else np.nan,
        'iterations': 0,
        'absorbed_or_collinear': '; '.join(c for c in names if c not in identified),
        'fixed_effects': '; '.join(fixed_effects),
    }
    coefficients = pd.DataFrame({
        'variable': identified, 'beta': estimates, 'se': se,
    })
    return (result, coefficients)



In [ ]:
rng=np.random.default_rng(SEED)
cluster_rows=[np.flatnonzero(group==g) for g in range(group.max()+1)]
boot=[];failures=[]
for rep in range(BOOTSTRAP_REPS):
    selected=rng.integers(0,len(cluster_rows),len(cluster_rows))
    idx=np.concatenate([cluster_rows[g] for g in selected])
    for category in MAIN:
        x,scales=designs[category]
        for tau in QUANTILES:
            try:
                beta,method,loss=fit_quantile(y[idx],x[idx],tau,max_iter=1)
                boot.append(dict(rep=rep,crime=category,quantile=tau,beta=beta[0]/scales[0],solver=method))
            except Exception as error:
                failures.append(dict(rep=rep,crime=category,quantile=tau,error=str(error)))
    if rep%5==0:print('Bootstrap',rep+1,'/',BOOTSTRAP_REPS,flush=True)
bootstrap=pd.DataFrame(boot)
bootstrap.to_csv(OUT/'cluster_bootstrap_draws.csv',index=False)
pd.DataFrame(failures,columns=['rep','crime','quantile','error']).to_csv(OUT/'bootstrap_failures.csv',index=False)
ci=bootstrap.groupby(['crime','quantile']).beta.agg(bootstrap_success='count',bootstrap_se='std',ci_low=lambda x:x.quantile(.025),ci_high=lambda x:x.quantile(.975)).reset_index()
results=points.merge(ci,on=['crime','quantile'],validate='one_to_one')
assert results.bootstrap_success.ge(.8*BOOTSTRAP_REPS).all(), 'Too many bootstrap failures; inspect failure log'
results.to_csv(OUT/'quantile_results.csv',index=False)
ols=[]
for category in MAIN:
    r,c=fit_absorbed(sample,'log1p_'+category,CONTROLS,['oslaua','yearquarter'])
    ols.append(dict(crime=category,**r))
ols=pd.DataFrame(ols);ols.to_csv(OUT/'matched_ols_benchmark.csv',index=False)
table(results,'quantile_results')


Bootstrap 1 / 39
Bootstrap 6 / 39
Bootstrap 11 / 39


## 3. Actual betas across quantiles, controls and bootstrap uncertainty

The dotted OLS line is a mean regression, not the median coefficient. Bootstrap intervals are pilot percentile intervals. A difference between quantiles is not established merely because one interval includes zero and another does not. The final table uses paired bootstrap differences, with each replication shared across quantiles.


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,5))
for ax,category in zip(axes,MAIN):
    b=results.loc[results.crime.eq(category)].sort_values('quantile')
    ax.plot(b['quantile'],b.beta,'o-',label='Quantile beta')
    ax.fill_between(b['quantile'],b.ci_low,b.ci_high,alpha=.2,label='Pilot cluster-bootstrap 95% interval')
    ax.axhline(0,color='black',linestyle='--')
    ax.axhline(ols.loc[ols.crime.eq(category),'beta'].iloc[0],color='#d17b0f',linestyle=':',label='Matched OLS mean')
    for row in b.itertuples():ax.annotate(f'{row.beta:+.3f}',(row.quantile,row.beta),xytext=(0,8),textcoords='offset points',ha='center')
    ax.set_xticks(QUANTILES);ax.set_title(CRIME_NAMES[category]);ax.set_xlabel('Conditional price quantile');ax.set_ylabel('Crime beta');ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(OUT/'crime_beta_by_quantile.png',dpi=170,bbox_inches='tight');plt.show()
coef=pd.concat(all_coefficients)
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,var in zip(axes,['detached_dum','freehold','newbuild']):
    for category in MAIN:
        b=coef.loc[coef.variable.eq(var)&coef.crime.eq(category)].sort_values('quantile')
        ax.plot(b['quantile'],100*np.expm1(b.beta),marker='o',label=CRIME_NAMES[category])
    ax.set_title(var);ax.set_ylabel('Conditional price premium (%)');ax.set_xlabel('Quantile');ax.legend()
fig.tight_layout();fig.savefig(OUT/'property_premiums_by_quantile.png',dpi=170);plt.show()
differences=[]
for category in MAIN:
    b=bootstrap.loc[bootstrap.crime.eq(category)].pivot(index='rep',columns='quantile',values='beta')
    delta=(b[.95]-b[.05]).dropna()
    p=points.loc[points.crime.eq(category)].set_index('quantile').beta
    differences.append(dict(crime=category,beta95_minus_beta05=p[.95]-p[.05],pilot_ci_low=delta.quantile(.025),pilot_ci_high=delta.quantile(.975),successful_pairs=len(delta)))
table(pd.DataFrame(differences),'tail_quantile_differences')


/tmp/ipykernel_3828922/480343534.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'crime_beta_by_quantile.png',dpi=170,bbox_inches='tight');plt.show()


/tmp/ipykernel_3828922/480343534.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'property_premiums_by_quantile.png',dpi=170);plt.show()


,crime,beta95_minus_beta05,pilot_ci_low,pilot_ci_high,successful_pairs
0,mps_all,0.0951,0.0350,0.1271,39
1,mps_violence,0.0377,-0.0169,0.0775,39
2,mps_property,0.0588,0.0156,0.1282,39


## 4. Every control coefficient and limitations

All coefficients, including borough/time dummies, are saved. The crime bootstrap intervals are not available for every control; do not infer control significance from the point-only table. The sampled model uses borough effects, not LSOA quantile fixed effects. It estimates conditional associations, not price-segment treatment effects or causal effects. Run larger samples and at least 199-499 cluster bootstrap replications before relying on a final inference claim. Neither quantile regression nor a spatial plot guarantees a negative crime coefficient.


In [ ]:
from IPython.display import HTML
cards=[]
for (category,tau),b in coef.groupby(['crime','quantile'],sort=False):
    cards.append('<details style="margin:10px;padding:8px;border:1px solid #bbb"><summary>'+CRIME_NAMES[category]+' / tau='+str(tau)+'</summary>'+b[['variable','beta']].to_html(index=False,float_format=lambda x:f'{x:.5f}')+'</details>')
display(HTML(''.join(cards)))


variable,beta
log1p_mps_all,-0.07076
Intercept,12.99617
detached_dum,0.52989
semi_d_dum,0.37093
terrace_dum,0.18172
freehold,0.33897
newbuild,0.15874
tube_distnear_km,-0.02792
bus_distnear_km,0.10327
thamesriv_dist_km,-0.00748


## 5. Quantile-ordering check

Separately estimated conditional quantiles can cross. This diagnostic checks predicted ordering on the analysis sample; it does not enforce ordering outside that sample.


In [ ]:
coefficient_frame=pd.read_csv(OUT/'all_quantile_coefficients.csv')
crossing_rows=[]
for category in MAIN:
    crime=sample['log1p_'+category].to_numpy(float);scale=max(crime.std(),1e-12)
    x=np.column_stack([crime/scale,xbase]);scales=np.r_[scale,base_scale]
    variable_order=['log1p_'+category]+names
    predicted=[]
    for tau in QUANTILES:
        beta=coefficient_frame.loc[coefficient_frame.crime.eq(category)&coefficient_frame['quantile'].eq(tau)].set_index('variable').reindex(variable_order).beta.to_numpy()
        predicted.append(x@(beta*scales))
    predictions=np.column_stack(predicted)
    crossed=np.any(np.diff(predictions,axis=1)<-1e-6,axis=1)
    crossing_rows.append(dict(crime=category,analysed_sales=len(sample),crossing_count=int(crossed.sum()),quantile_crossing_share=crossed.mean()))
table(pd.DataFrame(crossing_rows),'quantile_crossing_diagnostic')

,crime,analysed_sales,crossing_count,quantile_crossing_share
0,mps_all,6000,1,0.0002
1,mps_violence,6000,0,0.0000
2,mps_property,6000,0,0.0000


## 6. Findings from the saved run

The experiment uses 6,000 reproducibly sampled sales from 338,347 eligible sales, with 3,129 LSOA clusters. All 39 bootstrap replications succeeded for each of the 15 models.

| Crime | 5th-quantile beta | Median beta | 95th-quantile beta |
|---|---:|---:|---:|
| All crime | -0.07075 | -0.03509 | +0.02430 |
| Violence | -0.13221 | -0.13882 | -0.09455 |
| Property crime | +0.00732 | +0.02746 | +0.06613 |

The pilot percentile intervals exclude zero for all crime at the 5th, 25th and 50th quantiles, but include zero at the 75th and 95th. Violence is negative with intervals excluding zero throughout. Property crime has positive intervals excluding zero at the median and higher quantiles. Paired bootstrap differences between the 95th and 5th coefficients have intervals excluding zero for all crime and property crime, but not violence. These are preliminary results from 39 bootstrap draws, without multiple-testing adjustment.

The estimates use borough and quarter indicators, not LSOA quantile fixed effects; they cannot establish the within-LSOA causal relationships studied in M7. Different location controls, the limited property attributes and the small exploratory sample remain material limitations. The spatial notebook also detects dependence between LSOAs that this within-LSOA bootstrap does not cover.
